# Import thư viện và Đọc dữ liệu gốc

In [ ]:
#Khai báo thư viện
import pandas as pd
import numpy as np

In [ ]:
#Đọc file và và chuyển dạng chuỗi (string) trước để tự kiểm soát việc ép kiểu
reviews='/content/reviews.csv'
df_raw = pd.read_csv(reviews)
# , dtype=str, keep_default_na=True
df_raw.head()

,review_id,order_id,product_id,customer_id,review_date,rating,review_title
0,REV-0000001,1,2400,58578,2012-07-24,5,Highly recommend
1,REV-0000002,3,396,58811,2012-08-03,5,Very satisfied
2,REV-0000003,10,1431,49101,2012-07-23,5,Great quality
3,REV-0000005,16,1668,41028,2012-08-05,5,Great quality
4,REV-0000006,17,2352,42030,2012-07-17,4,Good overall


# Xem tổng quan

In [ ]:
#Đọc số dòng cột của file và xem kiểu dữ liệu của từng thuộc tính
df=df_raw.copy()
print("Số dòng và số cột",df.shape)
print("\nTên cột & dtype hiện tại (đọc dạng string để kiểm soát):")
print(df.dtypes)

Số dòng và số cột (113551, 7)

Tên cột & dtype hiện tại (đọc dạng string để kiểm soát):
review_id       object
order_id         int64
product_id       int64
customer_id      int64
review_date     object
rating           int64
review_title    object
dtype: object


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113551 entries, 0 to 113550
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   review_id     113551 non-null  object
 1   order_id      113551 non-null  int64 
 2   product_id    113551 non-null  int64 
 3   customer_id   113551 non-null  int64 
 4   review_date   113551 non-null  object
 5   rating        113551 non-null  int64 
 6   review_title  113551 non-null  object
dtypes: int64(4), object(3)
memory usage: 6.1+ MB


In [ ]:
df.describe()

,order_id,product_id,customer_id,rating
count,113551.000000,113551.000000,113551.000000,113551.000000
mean,408999.519740,1232.018705,85694.342762,3.936011
std,239021.922809,690.839232,48501.480918,1.149867
min,1.000000,3.000000,2.000000,1.000000
25%,202048.500000,689.000000,42096.000000,3.000000
50%,406841.000000,981.000000,89755.000000,4.000000
75%,614844.000000,2045.000000,133850.000000,5.000000
max,833296.000000,2412.000000,157563.000000,5.000000


# Định nghĩa các hàm dùng chung

In [ ]:
#thống kê và tìm kiếm các giá trị bất thường (outlier) bằng IQR
def iqr_outlier_stats(s):
    s = pd.to_numeric(s, errors='coerce')
    if s.empty:
      return 0, 0.0, None, None
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = s[(s < lo) | (s > hi)]
    # Trả về 4 thông số: Số lượng outlier, tỷ lệ %, ngưỡng dưới và ngưỡng trên.
    return len(outliers), round(len(outliers) / len(s) * 100, 3), lo, hi
#phát hiện các giá trị bất thường (outlier) bằng Z-score
from sklearn.preprocessing import StandardScaler
def get_zscore_outliers(s, thresh=3.0):
    s = pd.to_numeric(s, errors='coerce')
    if s.empty or s.std(ddof=0) == 0: return 0, 0.0
    scaler = StandardScaler()
    z_scores = scaler.fit_transform(s.values.reshape(-1, 1))
    return pd.Series(np.abs(z_scores.flatten()) > thresh, index=s.index)
#index=s.index: gắn lại đúng tên dòng (nhãn chỉ mục) của bảng dữ liệu gốc cho cột kết quả
# flatten(): chuyển mảng 2 chiều thành mảng 1 chiều
# reshape(-1, 1): -1:chọn tất cả giá trị và 1:chuyển về mảng 1 chiều
def normalize_person_name(x):
    """Trim khoảng trắng thừa (kể cả khoảng trắng ở giữa), viết hoa chữ cái đầu mỗi từ."""
    if pd.isna(x):
        return x
    x = " ".join(str(x).strip().split())
    return x.title() if x != "" else np.nan
def normalize_zip_code(x):
    """Chỉ giữ lại các ký tự số và trả về dưới dạng chuỗi (text), không ép độ dài."""
    if pd.isna(x):
        return x
    # Ép kiểu sang str và lọc lấy các ký tự là số
    digits = "".join(ch for ch in str(x).strip() if ch.isdigit())
    if digits == "":
        return np.nan
    # Trả về chuỗi kết quả nguyên bản
    return digits
def normalize_phone_vn(x, expected_len=9):
    """Số điện thoại VN chuẩn 10 số, bắt đầu bằng 0. Nếu nguồn chỉ có 9 số (mất số 0 đầu) thì bù lại."""
    if pd.isna(x):
        return x
    digits = "".join(ch for ch in str(x).strip() if ch.isdigit())
    if digits == "":
        return np.nan
    if len(digits) == expected_len and not digits.startswith("0"):
        digits = "0" + digits
    return digits
# 1. Rút gọn hàm làm sạch tiền tệ
def clean_currency_string(x):
    if pd.isna(x): return np.nan
    s = str(x)
    for tok in ['₫', 'VND', 'vnd', '$', 'USD', 'usd', ',', '%']:
        s = s.replace(tok, '')
    try:
        return float(s.strip())
    except ValueError:
        return np.nan
# Từ điển chữ số (có thể bổ sung thêm nếu cần)
WORD_NUMBER_MAP = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
    "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50
}
#Rút gọn hàm chuyển chữ/số thành float (dùng .get() thông minh hơn)
def words_or_number_to_float(x):
    if pd.isna(x): return np.nan
    s = str(x).lower()
    for tok in ["percent", "%%", "%"]:
        s = s.replace(tok, "")
    s = s.strip()
    # Tra cứu từ điển trước, nếu không có thì ép float, lỗi thì trả về NaN
    val = WORD_NUMBER_MAP.get(s)
    if val is None:
        try:
            val = float(s)
        except ValueError:
            return np.nan
    return val if val >= 0 else np.nan

# Kiểm tra trùng lặp và dữ liệu thiếu

In [ ]:
pd.isnull(df).sum()

,0
review_id,0
order_id,0
product_id,0
customer_id,0
review_date,0
rating,0
review_title,0


In [ ]:
#Kiểm tra tỷ lệ thiếu ở từng cột
# Viết một hàm nhỏ để lấy giá trị đầu tiên không bị trống
def lay_gia_tri_dau(cot):
    hop_le = cot.dropna()
    if len(hop_le) > 0:
        return hop_le.iloc[0]  # Lấy giá trị đầu tiên
    return None                # Nếu cột toàn rỗng thì trả về None

# Sau đó dùng hàm này cho bảng
profile_df = pd.DataFrame({
    'Cột': df.columns,
    'Kiểu dữ liệu': df.dtypes.values,
    'Số lượng thiếu': df.isna().sum().values,
    'Tỷ lệ thiếu (%)': (df.isna().mean() * 100).round(2).values,
    'Số giá trị unique': df.nunique().values,
    'Ví dụ mẫu': [lay_gia_tri_dau(df[c]) for c in df.columns]
})
profile_df

,Cột,Kiểu dữ liệu,Số lượng thiếu,Tỷ lệ thiếu (%),Số giá trị unique,Ví dụ mẫu
0,review_id,object,0,0.0,113551,REV-0000001
1,order_id,int64,0,0.0,111369,1
2,product_id,int64,0,0.0,1412,2400
3,customer_id,int64,0,0.0,48676,58578
4,review_date,object,0,0.0,3825,2012-07-24
5,rating,int64,0,0.0,5,5
6,review_title,object,0,0.0,18,Highly recommend


In [ ]:
cols = df.columns.to_list()
for c in cols:
    if c in df.columns:
        so_luong = (df[c].value_counts().sort_index(ascending=True) > 1).sum()
        print(f"Số lượng {c} xuất hiện nhiều hơn 1 lần là: {so_luong}")


Số lượng review_id xuất hiện nhiều hơn 1 lần là: 0
Số lượng order_id xuất hiện nhiều hơn 1 lần là: 2179
Số lượng product_id xuất hiện nhiều hơn 1 lần là: 1297
Số lượng customer_id xuất hiện nhiều hơn 1 lần là: 25688
Số lượng review_date xuất hiện nhiều hơn 1 lần là: 3823
Số lượng rating xuất hiện nhiều hơn 1 lần là: 5
Số lượng review_title xuất hiện nhiều hơn 1 lần là: 18


In [ ]:
for c in cols:
    if c in df.columns:
        counts = df[c].value_counts()
        frequent_values = counts[counts > 1].index
        print(f"giá trị của cột {c} xuất hiện nhiều hơn 1 lần là:",list(frequent_values))

giá trị của cột review_id xuất hiện nhiều hơn 1 lần là: []
giá trị của cột order_id xuất hiện nhiều hơn 1 lần là: [215573, 215503, 80530, 589788, 818125, 47044, 163611, 243990, 13351, 494254, 386929, 426049, 146236, 705962, 504953, 399788, 286271, 740380, 276755, 399765, 458695, 276852, 602055, 139044, 484489, 450120, 38213, 13153, 184174, 476858, 458734, 803371, 221972, 38221, 13221, 803389, 146302, 570148, 13270, 163570, 13199, 365396, 365398, 286386, 346495, 639813, 601379, 13769, 346415, 26256, 740208, 129705, 96783, 527621, 329294, 686143, 552970, 269897, 251919, 233196, 505208, 553125, 728713, 426085, 129820, 286300, 504888, 38315, 705763, 546647, 679031, 399654, 450259, 686184, 426200, 66622, 13641, 365391, 494363, 197593, 233764, 26820, 183855, 740812, 627201, 138895, 553604, 450097, 695474, 65976, 38038, 183933, 243858, 751316, 252758, 760402, 578922, 339343, 297189, 459369, 252984, 233887, 197829, 12439, 172575, 347331, 37982, 47571, 819007, 65812, 221345, 400735, 97861, 6271

In [ ]:
#Kiểm tra trùng lặp
n_dup_full = df.duplicated().sum()
n_dup_key = df.duplicated(subset=['review_id']).sum()
print(f"Số dòng trùng lặp hoàn toàn: {n_dup_full}")
print(f"Số dòng trùng theo khoá chính review_id: {n_dup_key}")
if n_dup_key > 0:
    df = df.drop_duplicates(subset=['review_id'], keep='first')
print("Kích thước sau khi loại trùng:", df.shape)

Số dòng trùng lặp hoàn toàn: 0
Số dòng trùng theo khoá chính review_id: 0
Kích thước sau khi loại trùng: (113551, 7)


# Chuẩn hoá định dạng & ép kiểu dữ liệu

In [ ]:
import numpy as np

# Chuẩn hoá format
df['review_id'] = df['review_id'].astype(str).str.strip()
df['review_title'] = df['review_title'].astype(str).str.strip()
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
# Lọc bỏ các giá trị rating ngoài phạm vi hợp lệ (từ 1 đến 5)
df.loc[(df['rating'] < 1) | (df['rating'] > 5), 'rating'] = np.nan
print("Giá trị rating sau chuẩn hoá:")
print('rating:', sorted(df['rating'].dropna().unique()))
df[['review_id', 'review_title', 'rating']].head()
# Chuyển kiểu
for c in ['order_id', 'product_id', 'customer_id', 'rating']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
df['review_date'] = pd.to_datetime(df['review_date'], errors='coerce')
for c in ['review_id', 'review_title']:
    if c in df.columns:
        df[c] = df[c].astype('string')

print(df.dtypes)

Giá trị rating sau chuẩn hoá:
rating: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
review_id       string[python]
order_id                 Int64
product_id               Int64
customer_id              Int64
review_date     datetime64[ns]
rating                   Int64
review_title    string[python]
dtype: object


# Kiểm tra logic nghiệp vụ cho từng cột

In [ ]:
print("\n--- Kiểm tra logic nghiệp vụ cho từng cột ---")

# 1. review_id: Phải là duy nhất
n_dup_review_id = df['review_id'].duplicated().sum()
if n_dup_review_id == 0:
    print(f"review_id: OK - Tất cả các giá trị đều là duy nhất.")
else:
    print(f"review_id: CẢNH BÁO - Có {n_dup_review_id} giá trị review_id trùng lặp.")

# 2. order_id, product_id, customer_id: Phải là số nguyên dương
id_cols = ['order_id', 'product_id', 'customer_id']
for col in id_cols:
    if col in df.columns:
        invalid_ids = df[df[col] <= 0]
        if len(invalid_ids) == 0:
            print(f"{col}: OK - Tất cả các giá trị đều là số nguyên dương.")
        else:
            print(f"{col}: CẢNH BÁO - Có {len(invalid_ids)} giá trị không phải là số nguyên dương (<= 0).")
# 3. review_date: Phải là ngày tháng hợp lệ và không được trong tương lai
if 'review_date' in df.columns:
    # Kiểm tra giá trị thiếu (NaN) sau khi chuyển đổi kiểu
    missing_dates = df['review_date'].isna().sum()
    if missing_dates == 0:
        print("review_date: OK - Không có giá trị ngày tháng nào bị thiếu hoặc không hợp lệ.")
    else:
        print(f"review_date: CẢNH BÁO - Có {missing_dates} giá trị ngày tháng bị thiếu hoặc không hợp lệ.")

    # Kiểm tra ngày trong tương lai
    today = pd.to_datetime('today')
    future_dates = df[df['review_date'] > today]
    if len(future_dates) == 0:
        print("review_date: OK - Không có giá trị ngày tháng nào trong tương lai.")
    else:
        print(f"review_date: CẢNH BÁO - Có {len(future_dates)} giá trị ngày tháng trong tương lai.")
# 4. rating: Phải nằm trong khoảng từ 1 đến 5 (đã được xử lý ở bước trước, kiểm tra lại)
if 'rating' in df.columns:
    invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5) | (df['rating'].isna())]
    if len(invalid_ratings) == 0:
        print(f"rating: OK - Tất cả các giá trị đều nằm trong khoảng từ 1 đến 5.")
    else:
        print(f"rating: CẢNH BÁO - Có {len(invalid_ratings)} giá trị rating không hợp lệ (ngoài 1-5 hoặc thiếu). Kiểm tra lại dữ liệu gốc nếu cần.")
# 5. review_title: Không được rỗng
if 'review_title' in df.columns:
    empty_titles = df[df['review_title'].str.strip() == '']
    if len(empty_titles) == 0:
        print(f"review_title: OK - Tất cả các tiêu đề đánh giá đều không rỗng.")
    else:
        print(f"review_title: CẢNH BÁO - Có {len(empty_titles)} tiêu đề đánh giá bị rỗng.")


--- Kiểm tra logic nghiệp vụ cho từng cột ---
review_id: OK - Tất cả các giá trị đều là duy nhất.
order_id: OK - Tất cả các giá trị đều là số nguyên dương.
product_id: OK - Tất cả các giá trị đều là số nguyên dương.
customer_id: OK - Tất cả các giá trị đều là số nguyên dương.
review_date: OK - Không có giá trị ngày tháng nào bị thiếu hoặc không hợp lệ.
review_date: OK - Không có giá trị ngày tháng nào trong tương lai.
rating: OK - Tất cả các giá trị đều nằm trong khoảng từ 1 đến 5.
review_title: OK - Tất cả các tiêu đề đánh giá đều không rỗng.


# Kiểm tra khóa ngoại

In [ ]:
print("--- Kiểm tra khóa ngoại cho bảng Reviews ---")
# 1. Đọc và kiểm tra tổng quan các bảng tham chiếu (Master Tables)
df_customers = pd.read_csv('/content/customers.csv')
df_orders = pd.read_csv('/content/orders_enriched.csv')
df_products = pd.read_csv('/content/products.csv')
# Ép kiểu dữ liệu an toàn cho các khóa chính của bảng tham chiếu
df_customers['customer_id'] = pd.to_numeric(df_customers['customer_id'], errors='coerce').astype('Int64')
df_orders['order_id'] = pd.to_numeric(df_orders['order_id'], errors='coerce').astype('Int64')
df_products['product_id'] = pd.to_numeric(df_products['product_id'], errors='coerce').astype('Int64')
# 2. Kiểm tra khóa ngoại cho 'customer_id'
if 'customer_id' in df.columns:
    customers_not_in_master = df[~df['customer_id'].isin(df_customers['customer_id'])]
    if not customers_not_in_master.empty:
        print(f"\nCó {len(customers_not_in_master)} dòng có `customer_id` trong `df` không tồn tại trong `customers.csv`.")
        print("Các `customer_id` không khớp (5 ví dụ đầu):", customers_not_in_master['customer_id'].dropna().unique()[:5].tolist())
    else:
        print("\nTất cả `customer_id` trong `df` đều khớp với `customers.csv`.")
# 3. Kiểm tra khóa ngoại cho 'order_id'
if 'order_id' in df.columns:
    orders_not_in_master = df[~df['order_id'].isin(df_orders['order_id'])]
    if not orders_not_in_master.empty:
        print(f"\nCó {len(orders_not_in_master)} dòng có `order_id` trong `df` không tồn tại trong `orders_enriched.csv`.")
        print("Các `order_id` không khớp (5 ví dụ đầu):", orders_not_in_master['order_id'].dropna().unique()[:5].tolist())
    else:
        print("\nTất cả `order_id` trong `df` đều khớp với `orders_enriched.csv`.")
# 4. Kiểm tra khóa ngoại cho 'product_id'
if 'product_id' in df.columns:
    products_not_in_master = df[~df['product_id'].isin(df_products['product_id'])]

    if not products_not_in_master.empty:
        print(f"\nCó {len(products_not_in_master)} dòng có `product_id` trong `df` không tồn tại trong `products.csv`.")
        print("Các `product_id` không khớp (5 ví dụ đầu):", products_not_in_master['product_id'].dropna().unique()[:5].tolist())
    else:
        print("\nTất cả `product_id` trong `df` đều khớp với `products.csv`.")

--- Kiểm tra khóa ngoại cho bảng Reviews ---

Tất cả `customer_id` trong `df` đều khớp với `customers.csv`.

Tất cả `order_id` trong `df` đều khớp với `orders_enriched.csv`.

Tất cả `product_id` trong `df` đều khớp với `products.csv`.


# Xuất file kết quả

In [ ]:
df.to_csv('reviews_silver.csv', index=False, encoding='utf-8-sig')
print("Đã lưu file đã làm sạch tại: reviews_silver.csv",)
print(f"Kích thước cuối cùng: {df.shape}")

Đã lưu file đã làm sạch tại: reviews_silver.csv
Kích thước cuối cùng: (113551, 7)
